In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Mount Google Drive and set shared paths
# Run this first, then run cells 2-4 in any order.
# ════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, glob

# ── Your Drive folder layout ─────────────────────────────────
# MyDrive/talking_head/
#   headshots/           person_01.png ... person_08.png
#   outputs/             results saved here automatically
#   template_talking.mp4 (optional — used if you uncomment it in Cell 2)

DRIVE_ROOT    = '/content/drive/MyDrive/talking_head'
HEADSHOTS_DIR = f'{DRIVE_ROOT}/headshots'
OUTPUTS_DIR   = f'{DRIVE_ROOT}/outputs'
TEMPLATE_MP4  = f'{DRIVE_ROOT}/template_talking.mp4'
MODELS_DIR    = '/content/models'   # local to Colab session (fast)

for d in [f'{OUTPUTS_DIR}/liveportrait',
          f'{OUTPUTS_DIR}/sadtalker',
          f'{OUTPUTS_DIR}/wav2lip']:
    os.makedirs(d, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

headshots = sorted(glob.glob(f'{HEADSHOTS_DIR}/*.png'))
print(f'Found {len(headshots)} headshots:')
for h in headshots:
    print(f'  {os.path.basename(h)}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — LivePortrait  (GPU, hybrid pkl template)
# ════════════════════════════════════════════════════════════
import os, sys, glob, pickle, subprocess, time
import numpy as np
from pathlib import Path

LP_REPO = Path(MODELS_DIR) / 'liveportrait'
LP_OUT  = Path(OUTPUTS_DIR) / 'liveportrait'
LP_OUT.mkdir(parents=True, exist_ok=True)

HF_REPO = 'KlingTeam/LivePortrait'
WEIGHT_FILES = [
    'liveportrait/base_models/appearance_feature_extractor.pth',
    'liveportrait/base_models/motion_extractor.pth',
    'liveportrait/base_models/spade_generator.pth',
    'liveportrait/base_models/warping_module.pth',
    'liveportrait/retargeting_models/stitching_retargeting_module.pth',
    'liveportrait/landmark.onnx',
]

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

def _safe_load_pkl(path):
    """
    Load pkl files created with older numpy.
    Uses a custom Unpickler to intercept numpy's _reconstruct lookup —
    more reliable than monkey-patching the module attribute (which numpy 2.0 bypasses).
    """
    class _Loader(pickle.Unpickler):
        def find_class(self, module, name):
            if name == '_reconstruct' and 'multiarray' in module:
                return _Loader._reconstruct
            return super().find_class(module, name)

        @staticmethod
        def _reconstruct(cls, *args):
            if not (isinstance(cls, type) and issubclass(cls, np.ndarray)):
                cls = np.ndarray
            alloc_shape = args[0] if args else (0,)
            typecode    = args[1] if len(args) > 1 else b'b'
            dtype = np.dtype(typecode) if isinstance(typecode, bytes) else np.dtype('b')
            return np.ndarray.__new__(cls, alloc_shape, dtype)

    with open(path, 'rb') as f:
        return _Loader(f).load()

# ── Setup (clone + install) ───────────────────────────────────
if not (LP_REPO / '.setup_done').exists():
    _step('SETUP — cloning LivePortrait + installing deps')
    if not LP_REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/KwaiVGI/LivePortrait', str(LP_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', str(LP_REPO / 'requirements_base.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'onnxruntime', 'transformers==4.38.0', 'huggingface_hub'], check=True)
    (LP_REPO / '.setup_done').touch()
    print('  Setup done.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Download weights (~600 MB, once) ─────────────────────────
if not (LP_REPO / '.weights_done').exists():
    _step('WEIGHTS — downloading 6 files from HuggingFace')
    from huggingface_hub import hf_hub_download
    weights_dir = LP_REPO / 'pretrained_weights'
    weights_dir.mkdir(parents=True, exist_ok=True)
    for f in WEIGHT_FILES:
        dest = weights_dir / f
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            print(f'  {Path(f).name} ...', end=' ', flush=True)
            t0 = time.time()
            hf_hub_download(repo_id=HF_REPO, filename=f, local_dir=str(weights_dir))
            print(f'{dest.stat().st_size/1024/1024:.0f} MB in {_fmt(time.time()-t0)}')
        else:
            print(f'  already have {Path(f).name}')
    (LP_REPO / '.weights_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Weights already downloaded, skipping.')

# ── Build hybrid driving template ────────────────────────────
# Hybrid = talking.pkl head/shoulder rotation + d8.pkl expressions
# Falls back to TEMPLATE_MP4 from Drive if old pkl files can't be loaded.
DRIVING_DIR = LP_REPO / 'assets' / 'examples' / 'driving'
hybrid_pkl  = DRIVING_DIR / 'talking_with_movement.pkl'

if not hybrid_pkl.exists():
    _step('DRIVING TEMPLATE — building hybrid or choosing fallback')
    try:
        talking = _safe_load_pkl(DRIVING_DIR / 'talking.pkl')
        d8      = _safe_load_pkl(DRIVING_DIR / 'd8.pkl')
        n      = talking['n_frames']
        idx    = np.linspace(0, len(d8['motion']) - 1, n).astype(int)
        d8_exp = np.stack([d8['motion'][i]['exp'] for i in idx])

        t_arr = np.linspace(0, 4 * np.pi, n)

        # Blink: wider open eyes, quick close
        blink = np.where(np.abs(np.sin(t_arr * 0.7)) > 0.97, 0.03, 0.28).astype(np.float32)

        # Lip: ~4 syllables/sec (20 cycles/5s), peak 0.40 to show teeth
        # phrase envelope varies amplitude like natural prosody
        syllable = np.abs(np.sin(t_arr * 5.0))
        envelope = 0.45 + 0.55 * np.abs(np.sin(t_arr * 0.8))
        lip      = (syllable * envelope * 0.40).astype(np.float32)
        lip      = np.where(lip < 0.05, 0.0, lip)   # hard-close (stop consonants)

        hybrid = {
            'n_frames': n, 'output_fps': talking['output_fps'],
            'motion': [{'scale': talking['motion'][i]['scale'],
                        'R_d':   talking['motion'][i]['R'],
                        'exp':   d8_exp[i],
                        't':     talking['motion'][i]['t']} for i in range(n)],
            'c_d_eyes_lst': [np.array([[b, b]], dtype=np.float32) for b in blink],
            'c_d_lip_lst':  [np.array([[float(lip[i])]], dtype=np.float32) for i in range(n)],
        }
        with open(hybrid_pkl, 'wb') as f: pickle.dump(hybrid, f)
        print(f'  Saved {hybrid_pkl.name}  ({n} frames @ {talking["output_fps"]}fps)')
        _driving = str(hybrid_pkl)
    except Exception as e:
        print(f'  pkl load failed ({e})')
        print(f'  Falling back to template_talking.mp4 from Drive')
        _driving = TEMPLATE_MP4
else:
    _driving = str(hybrid_pkl)
    print(f'[{time.strftime("%H:%M:%S")}] Hybrid template already built, skipping.')

# ── Inference ─────────────────────────────────────────────────
driving = _driving
print(f'\n  Using driver: {Path(driving).name}')

_step(f'INFERENCE — {len(headshots)} headshots  |  multiplier 1.2')
total_start = time.time()

for headshot in headshots:
    name     = Path(headshot).stem
    out_path = str(LP_OUT / f'{name}_talking.mp4')
    if os.path.exists(out_path):
        print(f'  skip {name} (already exists)')
        continue
    work_dir = str(LP_OUT / name)
    os.makedirs(work_dir, exist_ok=True)
    cmd = [sys.executable, str(LP_REPO / 'inference.py'),
           '--source', headshot, '--driving', driving,
           '--output_dir', work_dir,
           '--driving_multiplier', '1.2']   # amplify for more expressive motion
    if driving.endswith('.mp4'):
        cmd.append('--flag_crop_driving_video')
    print(f'  [{time.strftime("%H:%M:%S")}] {name} ...', end=' ', flush=True)
    t0     = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(LP_REPO),
                            env={**os.environ, 'PYTHONUTF8': '1'})
    elapsed = time.time() - t0
    if result.returncode != 0:
        print(f'ERROR\n{result.stderr[-400:]}')
        continue
    mp4s = sorted(glob.glob(os.path.join(work_dir, '**', '*.mp4'), recursive=True),
                  key=os.path.getmtime, reverse=True)
    if mp4s:
        os.replace(mp4s[0], out_path)
        print(f'done in {_fmt(elapsed)}  ({os.path.getsize(out_path)/1024/1024:.2f} MB)')
    else:
        print(f'WARNING: no mp4 found')

print(f'\n{"="*50}')
print(f'LivePortrait done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {LP_OUT}')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — SadTalker  (GPU, real TTS audio driver via edge-tts)
# ════════════════════════════════════════════════════════════
import os, sys, glob, shutil, subprocess, time, tempfile
import numpy as np
from pathlib import Path

ST_REPO         = Path(MODELS_DIR) / 'sadtalker'
ST_OUT          = Path(OUTPUTS_DIR) / 'sadtalker'
ST_CHECKPOINTS  = ST_REPO / 'checkpoints'
ST_GFPGAN       = ST_REPO / 'gfpgan' / 'weights'
ST_OUT.mkdir(parents=True, exist_ok=True)

_ST_BASE = 'https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc'
CHECKPOINTS = {
    'SadTalker_V0.0.2_256.safetensors': f'{_ST_BASE}/SadTalker_V0.0.2_256.safetensors',
    'SadTalker_V0.0.2_512.safetensors': f'{_ST_BASE}/SadTalker_V0.0.2_512.safetensors',
    'mapping_00229-model.pth.tar':       f'{_ST_BASE}/mapping_00229-model.pth.tar',
    'mapping_00109-model.pth.tar':       f'{_ST_BASE}/mapping_00109-model.pth.tar',
}
GFPGAN_WEIGHTS = {
    'alignment_WFLW_4HG.pth':      'https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth',
    'detection_Resnet50_Final.pth': 'https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth',
    'GFPGANv1.4.pth':              'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth',
    'parsing_parsenet.pth':         'https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth',
}

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

# ── Setup ─────────────────────────────────────────────────────
if not (ST_REPO / '.setup_done').exists():
    _step('SETUP — cloning SadTalker + installing deps')
    if not ST_REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/OpenTalker/SadTalker', str(ST_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'librosa', 'resampy', 'pydub', 'yacs', 'kornia', 'face_alignment',
                    'basicsr', 'gfpgan', 'safetensors', 'av',
                    'scikit-image', 'imageio', 'imageio-ffmpeg'], check=True)
    (ST_REPO / '.setup_done').touch()
    print('  Setup done.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Download checkpoints (~300 MB, once) ─────────────────────
if not (ST_REPO / '.checkpoints_done').exists():
    import urllib.request
    _step('CHECKPOINTS — downloading from GitHub Releases')
    ST_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    ST_GFPGAN.mkdir(parents=True, exist_ok=True)
    for fname, url in {**CHECKPOINTS, **GFPGAN_WEIGHTS}.items():
        dest = (ST_CHECKPOINTS if fname in CHECKPOINTS else ST_GFPGAN) / fname
        if not dest.exists():
            print(f'  {fname} ...', end=' ', flush=True)
            t0 = time.time()
            urllib.request.urlretrieve(url, str(dest))
            print(f'{dest.stat().st_size/1024/1024:.0f} MB in {_fmt(time.time()-t0)}')
        else:
            print(f'  already have {fname}')
    (ST_REPO / '.checkpoints_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Checkpoints already downloaded, skipping.')

# ── Compatibility patches ─────────────────────────────────────
import site
for sp in site.getsitepackages():
    target = Path(sp) / 'basicsr' / 'data' / 'degradations.py'
    if target.exists():
        txt = target.read_text(encoding='utf-8')
        OLD = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
        if OLD in txt:
            target.write_text(txt.replace(OLD, 'from torchvision.transforms.functional import rgb_to_grayscale'), encoding='utf-8')
            print('  Patched basicsr degradations.py')
        break
for fpath, replacements in [
    (ST_REPO / 'src/face3d/util/my_awing_arch.py', [('np.float,','float,'),('np.float)','float)')]),
    (ST_REPO / 'src/face3d/util/preprocess.py',    [('np.array([w0, h0, s, t[0], t[1]])',
                                                      'np.array([w0, h0, s, float(t[0]), float(t[1])])')]),
]:
    if fpath.exists():
        txt = fpath.read_text(encoding='utf-8'); orig = txt
        for old, new in replacements: txt = txt.replace(old, new)
        if txt != orig:
            fpath.write_text(txt, encoding='utf-8')
            print(f'  Patched {fpath.name}')

# ── Generate TTS driver audio (once, reused for all headshots) ──
# SadTalker was trained on real speech mel spectrograms.
# Synthetic signals never match those patterns — only real TTS produces
# the mel features needed for proper mouth animation.
_DRIVER_WAV  = '/tmp/sadtalker_driver.wav'
_DRIVER_TEXT = (
    "Hello, I am your virtual assistant. I am happy to help you today. "
    "Please feel free to ask me any questions you may have. "
    "I will do my best to assist you."
)
_DRIVER_VOICE = 'en-US-JennyNeural'

if not os.path.exists(_DRIVER_WAV):
    _step('TTS DRIVER — generating speech with edge-tts (Microsoft Neural TTS, free)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts', 'nest_asyncio'],
                   check=True)
    import asyncio, nest_asyncio, edge_tts
    nest_asyncio.apply()   # allows asyncio.run inside Jupyter's existing event loop

    _mp3 = '/tmp/sadtalker_driver.mp3'
    async def _gen():
        await edge_tts.Communicate(_DRIVER_TEXT, _DRIVER_VOICE).save(_mp3)
    asyncio.get_event_loop().run_until_complete(_gen())

    # Convert to 16 kHz mono WAV — SadTalker's expected format
    subprocess.run(['ffmpeg', '-y', '-i', _mp3, '-ar', '16000', '-ac', '1', _DRIVER_WAV],
                   capture_output=True, check=True)
    dur = os.path.getsize(_DRIVER_WAV) / (16000 * 2)   # bytes / (sr * 2 bytes/sample)
    print(f'  Driver audio ready: {dur:.1f}s  ({os.path.getsize(_DRIVER_WAV)//1024} KB)')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Driver audio already exists, skipping.')

# ── Inference ─────────────────────────────────────────────────
_step(f'INFERENCE — {len(headshots)} headshots  |  GPU  |  edge-tts driver')
total_start = time.time()

for headshot in headshots:
    name     = Path(headshot).stem
    out_path = str(ST_OUT / f'{name}_talking.mp4')
    if os.path.exists(out_path):
        print(f'  skip {name} (already exists)')
        continue

    result_dir = str(ST_REPO / 'results' / name)

    cmd = [sys.executable, str(ST_REPO / 'inference.py'),
           '--driven_audio', _DRIVER_WAV,
           '--source_image', headshot,
           '--result_dir',   result_dir,
           '--still', '--preprocess', 'full']
    # cmd += ['--enhancer', 'gfpgan']  # uncomment for GFPGAN face restoration

    print(f'  [{time.strftime("%H:%M:%S")}] {name} ...', end=' ', flush=True)
    t0     = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ST_REPO))
    elapsed = time.time() - t0

    if result.returncode != 0:
        print(f'ERROR\n{result.stderr[-600:]}')
        continue

    candidates = glob.glob(os.path.join(result_dir, '**', '*.mp4'), recursive=True)
    if candidates:
        shutil.copy2(max(candidates, key=os.path.getmtime), out_path)
        print(f'done in {_fmt(elapsed)}  ({os.path.getsize(out_path)/1024/1024:.2f} MB)')
    else:
        print(f'WARNING: no mp4 found')

print(f'\n{"="*50}')
print(f'SadTalker done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {ST_OUT}')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Wav2Lip  (GPU, edge-tts audio, FFmpeg sharpen)
# ════════════════════════════════════════════════════════════
import os, sys, glob, shutil, subprocess, time, tempfile
import numpy as np
from pathlib import Path

W2L_REPO       = Path(MODELS_DIR) / 'wav2lip'
W2L_OUT        = Path(OUTPUTS_DIR) / 'wav2lip'
W2L_CHECKPOINT = W2L_REPO / 'checkpoints' / 'wav2lip_gan.pth'
W2L_OUT.mkdir(parents=True, exist_ok=True)

_CHECKPOINT_URL = 'https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth'

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

# ── Setup ─────────────────────────────────────────────────────
if not (W2L_REPO / '.setup_done').exists():
    _step('SETUP — cloning Wav2Lip + installing deps')
    if not W2L_REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Rudrabha/Wav2Lip', str(W2L_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'librosa', 'face_alignment'], check=True)
    (W2L_REPO / '.setup_done').touch()
    print('  Setup done.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Download checkpoint (~400 MB, once) ───────────────────────
if not (W2L_REPO / '.checkpoint_done').exists():
    import urllib.request
    _step('CHECKPOINT — downloading wav2lip_gan.pth (~400 MB)')
    W2L_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    if not W2L_CHECKPOINT.exists():
        print('  downloading ...', end=' ', flush=True)
        t0 = time.time()
        urllib.request.urlretrieve(_CHECKPOINT_URL, str(W2L_CHECKPOINT))
        print(f'{W2L_CHECKPOINT.stat().st_size/1024/1024:.0f} MB in {_fmt(time.time()-t0)}')
    (W2L_REPO / '.checkpoint_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Checkpoint already downloaded, skipping.')

# ── Compatibility patch ───────────────────────────────────────
audio_py = W2L_REPO / 'audio.py'
if audio_py.exists():
    txt = audio_py.read_text(encoding='utf-8')
    OLD = 'return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,'
    NEW = 'return librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft, n_mels=hp.num_mels,'
    if OLD in txt:
        audio_py.write_text(txt.replace(OLD, NEW), encoding='utf-8')
        print('  Patched audio.py (librosa 0.9+ compat)')

# ── TTS driver audio ──────────────────────────────────────────
_DRIVER_WAV   = '/tmp/sadtalker_driver.wav'
_DRIVER_TEXT  = (
    "Hello, I am your virtual assistant. I am happy to help you today. "
    "Please feel free to ask me any questions you may have. "
    "I will do my best to assist you."
)
_DRIVER_VOICE = 'en-US-JennyNeural'

if not os.path.exists(_DRIVER_WAV):
    _step('TTS DRIVER — generating speech with edge-tts')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts', 'nest_asyncio'], check=True)
    import asyncio, nest_asyncio, edge_tts
    nest_asyncio.apply()
    _mp3 = '/tmp/w2l_driver.mp3'
    async def _gen():
        await edge_tts.Communicate(_DRIVER_TEXT, _DRIVER_VOICE).save(_mp3)
    asyncio.get_event_loop().run_until_complete(_gen())
    subprocess.run(['ffmpeg', '-y', '-i', _mp3, '-ar', '16000', '-ac', '1', _DRIVER_WAV],
                   capture_output=True, check=True)
    print(f'  Driver audio ready ({os.path.getsize(_DRIVER_WAV)//1024} KB)')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Reusing driver audio from Cell 3.')

# ── Sharpen post-pass (fixes Wav2Lip blurry mouth patch) ─────
def _sharpen(input_mp4, output_mp4):
    """
    FFmpeg unsharp mask on the Wav2Lip output.
    Wav2Lip pastes a low-res generated mouth region (96x96) back onto the
    original frame, leaving a blur seam. unsharp sharpens the entire frame
    including the pasted region, making the seam far less visible.
    No Python dependencies — avoids the numpy 2.x / gfpgan binary incompatibility.
    """
    r = subprocess.run([
        'ffmpeg', '-y', '-i', input_mp4,
        '-vf', 'unsharp=luma_msize_x=5:luma_msize_y=5:luma_amount=1.5',
        '-c:a', 'copy', '-crf', '18', '-preset', 'fast',
        output_mp4
    ], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-300:])

# ── Inference ─────────────────────────────────────────────────
_step(f'INFERENCE — {len(headshots)} headshots  |  GPU  |  25fps  |  unsharp post-pass')
total_start = time.time()
audio_dur   = os.path.getsize(_DRIVER_WAV) / (16000 * 2)

for headshot in headshots:
    name     = Path(headshot).stem
    out_path = str(W2L_OUT / f'{name}_talking.mp4')
    if os.path.exists(out_path):
        print(f'  skip {name} (already exists)')
        continue

    # Step 1: static image → looped video matching audio length
    tmp_vid = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
    subprocess.run(['ffmpeg', '-y', '-loop', '1', '-i', headshot,
                    '-t', str(audio_dur),
                    '-vf', 'fps=25,scale=trunc(iw/2)*2:trunc(ih/2)*2',
                    '-pix_fmt', 'yuv420p', tmp_vid],
                   capture_output=True, check=True)

    # Step 2: Wav2Lip inference → raw output with blurry mouth patch
    raw_out = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
    cmd = [sys.executable, str(W2L_REPO / 'inference.py'),
           '--checkpoint_path', str(W2L_CHECKPOINT),
           '--face',    tmp_vid,
           '--audio',   _DRIVER_WAV,
           '--outfile', raw_out,
           '--fps',     '25',
           '--nosmooth']

    print(f'  [{time.strftime("%H:%M:%S")}] {name} — Wav2Lip ...', end=' ', flush=True)
    t0     = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(W2L_REPO))

    try: os.unlink(tmp_vid)
    except OSError: pass

    if result.returncode != 0:
        print(f'ERROR\n{result.stderr[-600:]}')
        try: os.unlink(raw_out)
        except OSError: pass
        continue

    print(f'done ({_fmt(time.time()-t0)}) | sharpen ...', end=' ', flush=True)

    # Step 3: FFmpeg unsharp pass — reduces blurry mouth seam
    t1 = time.time()
    try:
        _sharpen(raw_out, out_path)
        print(f'done ({_fmt(time.time()-t1)})  →  {os.path.getsize(out_path)/1024/1024:.2f} MB')
    except Exception as e:
        shutil.copy2(raw_out, out_path)
        print(f'sharpen failed ({e}), kept raw')
    finally:
        try: os.unlink(raw_out)
        except OSError: pass

print(f'\n{"="*50}')
print(f'Wav2Lip done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {W2L_OUT}')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — EchoMimic V2  (AAAI 2025, audio-driven, half-body motion)
# https://github.com/antgroup/echomimic_v2
#
# IMPORTANT — this model's quirks (learned the hard way):
#   • Needs a driving POSE sequence (.npy per frame), not just audio.
#     We use the repo's bundled assets/halfbody_demo/pose/01 (768px).
#   • Pose render canvas is intrinsically 768x768 (baked into the .npy
#     draw_pose_params) → inference MUST run at W=H=768, not 512.
#   • from moviepy.editor → pygame → old pkg_resources crashes on Py3.12;
#     we patch infer_acc.py + the Debian pkg_resources file.
# ════════════════════════════════════════════════════════════
import os, sys, re, glob, shutil, subprocess, time
import urllib.request
import yaml as _yaml
from pathlib import Path

ECHO_REPO    = Path(MODELS_DIR) / 'echomimic_v2'
ECHO_OUT     = Path(OUTPUTS_DIR) / 'echomimic'
ECHO_WEIGHTS = ECHO_REPO / 'pretrained_weights'
ECHO_OUT.mkdir(parents=True, exist_ok=True)

# Inference resolution MUST match the bundled pose data (768px).
ECHO_RES    = 768
# Cap clip length — pose dir has 336 frames (~13s); 6s is plenty for a
# loopable demo and keeps T4 runtime/memory bounded across 8 headshots.
ECHO_MAX_FRAMES = 150

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

# ── Detect CUDA version ────────────────────────────────────────
def _detect_cuda_tag():
    sources = [('nvidia-smi', True), ('nvcc --version', True),
               ('/usr/bin/nvidia-smi', False), ('/usr/local/cuda/bin/nvcc --version', True)]
    for cmd, use_shell in sources:
        try:
            out = subprocess.run(cmd, shell=use_shell, capture_output=True, text=True, timeout=5).stdout
            m = re.search(r'(?:release |CUDA Version: )(\d+)\.(\d+)', out)
            if m:
                major, minor = int(m.group(1)), int(m.group(2))
                print(f'  System CUDA: {major}.{minor}', flush=True)
                if major >= 12: return 'cu124' if minor >= 4 else 'cu121'
                elif major == 11 and minor >= 8: return 'cu118'
                else: return 'cu121'
        except Exception: continue
    try:
        ver = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'],
                             capture_output=True, text=True).stdout.strip()
        m = re.search(r'\+(cu\d+)', ver)
        if m: print(f'  Inferred from torch: {m.group(1)}', flush=True); return m.group(1)
    except Exception: pass
    print('  CUDA detection failed — defaulting to cu121', flush=True)
    return 'cu121'

_CUDA_TAG  = _detect_cuda_tag()
_TORCH_IDX = f'https://download.pytorch.org/whl/{_CUDA_TAG}'
print(f'  PyTorch index: {_TORCH_IDX}')

def _install_cuda_torch():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--force-reinstall', '--no-deps',
                    'torch', 'torchvision', 'torchaudio',
                    '--index-url', _TORCH_IDX], check=True)

def _cuda_ok():
    return subprocess.run(
        [sys.executable, '-c', 'import torch; exit(0 if torch.cuda.is_available() else 1)'],
        capture_output=True).returncode == 0

# ── Pre-flight ────────────────────────────────────────────────
if not _cuda_ok():
    _step(f'CPU-only torch — force-reinstalling ({_CUDA_TAG})')
    _install_cuda_torch()
    if _cuda_ok():
        print('  CUDA torch verified OK.')
    else:
        raise RuntimeError(
            f'CUDA still unavailable after installing {_CUDA_TAG} torch.\n'
            'Check: Runtime → Change runtime type → must be T4 GPU (not CPU).')
else:
    _info = subprocess.run(
        [sys.executable, '-c',
         'import torch; print(torch.__version__, "on", torch.cuda.get_device_name(0))'],
        capture_output=True, text=True).stdout.strip()
    print(f'[{time.strftime("%H:%M:%S")}] CUDA torch OK — {_info}')

# ── Python 3.12 pkg_resources fix (run every session) ─────────
# Two broken pkg_resources locations on Colab:
#   1. /usr/local/.../dist-packages/  — fixed by upgrading setuptools via pip
#   2. /usr/lib/python3/dist-packages/ — Debian system, NOT touched by pip
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'setuptools'],
               check=True)
_debian_pkgres = Path('/usr/lib/python3/dist-packages/pkg_resources/__init__.py')
if _debian_pkgres.exists():
    _dr_txt = _debian_pkgres.read_text()
    _DR_OLD = 'register_finder(pkgutil.ImpImporter, find_on_path)'
    _DR_NEW = 'if hasattr(pkgutil, "ImpImporter"): register_finder(pkgutil.ImpImporter, find_on_path)'
    if _DR_OLD in _dr_txt:
        _debian_pkgres.write_text(_dr_txt.replace(_DR_OLD, _DR_NEW))
        print(f'[{time.strftime("%H:%M:%S")}] Patched Debian pkg_resources (ImpImporter guard).')
    else:
        print(f'[{time.strftime("%H:%M:%S")}] Debian pkg_resources already patched.')
print(f'[{time.strftime("%H:%M:%S")}] setuptools/pkg_resources OK.')

# ── Setup ──────────────────────────────────────────────────────
if not (ECHO_REPO / '.setup_done').exists():
    _step('SETUP — cloning EchoMimic V2 + installing deps')
    if not ECHO_REPO.exists():
        subprocess.run(['git', 'clone',
                        'https://github.com/antgroup/echomimic_v2', str(ECHO_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'diffusers==0.31.0', 'transformers', 'accelerate',
                    'omegaconf', 'decord', 'librosa',
                    'einops', 'imageio', 'imageio-ffmpeg',
                    'onnxruntime-gpu', 'mediapipe', 'moviepy==1.0.3'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--no-deps', 'git+https://github.com/openai/CLIP.git'], check=True)
    if (ECHO_REPO / 'requirements.txt').exists():
        _gpu_pkgs = {'torch', 'torchvision', 'torchaudio', 'xformers', 'torchao'}
        _reqs = [r.strip() for r in (ECHO_REPO / 'requirements.txt').read_text().splitlines()
                 if r.strip() and not r.startswith('#')
                 and not any(r.lower().startswith(p) for p in _gpu_pkgs)]
        if _reqs:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + _reqs, check=False)
    _step(f'SETUP finalising — pinning CUDA torch ({_CUDA_TAG})')
    _install_cuda_torch()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--force-reinstall', '--no-deps', 'xformers',
                    '--index-url', _TORCH_IDX], check=False)
    (ECHO_REPO / '.setup_done').touch()
    print('  Setup done (CUDA torch locked in).')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Patch infer_acc.py (run every session, idempotent) ────────
# moviepy.editor unconditionally imports pygame → old pkg_resources → Py3.12 crash.
# Replace with direct submodule imports that skip the preview.py → pygame chain.
_infer_acc = ECHO_REPO / 'infer_acc.py'
if _infer_acc.exists():
    _ia_txt = _infer_acc.read_text()
    _IA_OLD = 'from moviepy.editor import VideoFileClip, AudioFileClip'
    _IA_NEW = ('from moviepy.video.io.VideoFileClip import VideoFileClip\n'
               'from moviepy.audio.io.AudioFileClip import AudioFileClip')
    if _IA_OLD in _ia_txt:
        _infer_acc.write_text(_ia_txt.replace(_IA_OLD, _IA_NEW))
        print(f'[{time.strftime("%H:%M:%S")}] Patched infer_acc.py: moviepy.editor → direct imports.')
    else:
        print(f'[{time.strftime("%H:%M:%S")}] infer_acc.py already patched.')

# ── Download weights (~7 GB) ───────────────────────────────────
if not (ECHO_REPO / '.weights_done').exists():
    from huggingface_hub import snapshot_download
    _step('WEIGHTS — downloading EchoMimicV2 + SD models (~7 GB total)')
    ECHO_WEIGHTS.mkdir(parents=True, exist_ok=True)
    for hf_id, local_name in [
        ('BadToBest/EchoMimicV2',                    'EchoMimicV2'),
        ('stabilityai/sd-vae-ft-mse',                'sd-vae-ft-mse'),
        ('lambdalabs/sd-image-variations-diffusers', 'sd-image-variations-diffusers'),
    ]:
        dest = ECHO_WEIGHTS / local_name
        if not dest.exists():
            print(f'  {local_name} ...', end=' ', flush=True)
            t0 = time.time()
            snapshot_download(hf_id, local_dir=str(dest),
                              ignore_patterns=['*.ckpt', 'safety_checker*', 'flax*', 'tf_*'])
            print(f'done in {_fmt(time.time()-t0)}')
        else:
            print(f'  already have {local_name}')
    src = ECHO_WEIGHTS / 'EchoMimicV2'
    if src.exists():
        for f in src.iterdir():
            dst = ECHO_WEIGHTS / f.name
            if not dst.exists():
                (shutil.copytree if f.is_dir() else shutil.copy2)(str(f), str(dst))
    (ECHO_REPO / '.weights_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Weights already downloaded, skipping.')

# ── Whisper tiny model (not in HF repo — downloaded separately) ─
_TINY_PT = ECHO_WEIGHTS / 'audio_processor' / 'tiny.pt'
if not _TINY_PT.exists():
    _step('AUDIO PROCESSOR — downloading Whisper tiny model (~70 MB)')
    _TINY_PT.parent.mkdir(parents=True, exist_ok=True)
    _WHISPER_URL = ('https://openaipublic.azureedge.net/main/whisper/models/'
                    '65147644a518d12f04e32d6f3b26facc3f8dd46e5390956a9424a650c0ce22b9/tiny.pt')
    print('  downloading tiny.pt ...', end=' ', flush=True)
    t0 = time.time()
    urllib.request.urlretrieve(_WHISPER_URL, str(_TINY_PT))
    print(f'done in {_fmt(time.time()-t0)} ({_TINY_PT.stat().st_size//1024//1024} MB)')
else:
    print(f'[{time.strftime("%H:%M:%S")}] audio_processor/tiny.pt already exists.')

# ── Pose directory — bundled .npy sequences (cloned with repo) ─
# assets/halfbody_demo/pose/{01,02,03,04,fight,good,salute,ultraman}/
# Each contains numbered .npy files (0.npy … N.npy), one per frame,
# extracted for 768x768 output (drives W=H=768 above).
_POSE_DIR = ECHO_REPO / 'assets' / 'halfbody_demo' / 'pose' / '01'
if not _POSE_DIR.exists() or not any(_POSE_DIR.glob('*.npy')):
    raise RuntimeError(
        f'Pose directory missing or empty: {_POSE_DIR}\n'
        'Delete .setup_done and re-run to re-clone the repo.')
_POSE_FRAMES = len(list(_POSE_DIR.glob('*.npy')))
_GEN_FRAMES  = min(_POSE_FRAMES, ECHO_MAX_FRAMES)
print(f'[{time.strftime("%H:%M:%S")}] Pose dir: {_POSE_DIR.name}  '
      f'({_POSE_FRAMES} frames available, generating {_GEN_FRAMES} @ {ECHO_RES}px)')

# ── TTS driver audio ───────────────────────────────────────────
_DRIVER_WAV = '/tmp/sadtalker_driver.wav'
if not os.path.exists(_DRIVER_WAV):
    _step('TTS DRIVER — generating speech with edge-tts')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts', 'nest_asyncio'], check=True)
    import asyncio, nest_asyncio, edge_tts
    nest_asyncio.apply()
    async def _gen():
        await edge_tts.Communicate(
            "Hello, I am your virtual assistant. I am happy to help you today. "
            "Please feel free to ask me any questions you may have. "
            "I will do my best to assist you.", 'en-US-JennyNeural').save('/tmp/echo_drv.mp3')
    asyncio.get_event_loop().run_until_complete(_gen())
    subprocess.run(['ffmpeg', '-y', '-i', '/tmp/echo_drv.mp3',
                    '-ar', '16000', '-ac', '1', _DRIVER_WAV],
                   capture_output=True, check=True)
    print(f'  Driver audio ready.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Reusing driver audio.')

# ── Inference ──────────────────────────────────────────────────
_step(f'INFERENCE — {len(headshots)} headshots  |  GPU ({_CUDA_TAG})  |  '
      f'EchoMimic V2 acc  |  {ECHO_RES}px  |  {_GEN_FRAMES}f')
total_start = time.time()

# test_cases: {ref_image_path: [audio_path, pose_dir_path]}
# infer_acc.py distinguishes them: ".wav" in path → audio_path, else → pose_dir
cfg_path = '/tmp/echomimic_v2_batch.yaml'
_yaml.dump({
    'pretrained_base_model_path': str(ECHO_WEIGHTS / 'sd-image-variations-diffusers'),
    'pretrained_vae_path':        str(ECHO_WEIGHTS / 'sd-vae-ft-mse'),
    'denoising_unet_path':        str(ECHO_WEIGHTS / 'denoising_unet_acc.pth'),
    'reference_unet_path':        str(ECHO_WEIGHTS / 'reference_unet.pth'),
    'pose_encoder_path':          str(ECHO_WEIGHTS / 'pose_encoder.pth'),
    'motion_module_path':         str(ECHO_WEIGHTS / 'motion_module_acc.pth'),
    'audio_mapper_path':          str(ECHO_WEIGHTS / 'audio_mapper-50000.pth'),
    'audio_model_path':           str(_TINY_PT),
    'inference_config':           str(ECHO_REPO / 'configs' / 'inference' / 'inference_v2.yaml'),
    'weight_dtype': 'fp16',
    'seed': 2024, 'fps': 25, 'H': ECHO_RES, 'W': ECHO_RES, 'steps': 6,
    'test_cases': {hs: [_DRIVER_WAV, str(_POSE_DIR)] for hs in headshots},
}, open(cfg_path, 'w'))

cmd = [sys.executable, str(ECHO_REPO / 'infer_acc.py'),
       '--config', cfg_path,
       '-W', str(ECHO_RES), '-H', str(ECHO_RES),
       '-L', str(_GEN_FRAMES),
       '--seed', '2024', '--fps', '25', '--steps', '6']

print(f'  [{time.strftime("%H:%M:%S")}] batch inference ({len(headshots)} images, '
      f'{_GEN_FRAMES} frames each @ {ECHO_RES}px) ...', flush=True)
t0     = time.time()
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ECHO_REPO))
elapsed = time.time() - t0

if result.returncode != 0:
    print(f'ERROR:\n{result.stderr[-1500:]}')
else:
    # infer_acc.py writes: output/{date}/{time}--step_N-WxH--cfg_X/{ref_stem}/*_sig.mp4
    # relative to ECHO_REPO (cwd during inference)
    all_sig = glob.glob(str(ECHO_REPO / 'output' / '**' / '*_sig.mp4'), recursive=True)
    for headshot in headshots:
        name     = Path(headshot).stem
        out_path = str(ECHO_OUT / f'{name}_talking.mp4')
        if os.path.exists(out_path):
            print(f'  {name}: already exists')
            continue
        matches = [p for p in all_sig if Path(p).parent.name == name]
        if not matches:
            matches = [p for p in all_sig if Path(p).stem.startswith(name)]
        if matches:
            best = max(matches, key=os.path.getmtime)
            shutil.copy2(best, out_path)
            print(f'  {name}: {os.path.getsize(out_path)/1024/1024:.2f} MB')
        else:
            print(f'  {name}: WARNING — no output found in ECHO_REPO/output/**/*_sig.mp4')
    print(f'  Inference time: {_fmt(elapsed)}')

print(f'\n{"="*50}')
print(f'EchoMimic V2 done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {ECHO_OUT}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — MuseTalk  (latent diffusion lip sync, no blur seam)
# https://github.com/TMElyralab/MuseTalk
#
# OpenMMLab without the pain: Colab has NO nvcc (can't compile mmcv) and
# mmcv 2.0.1 has no cp312 wheel. BUT mmcv 2.2.0 ships a prebuilt cp312
# wheel for torch2.4.0/cu121. So we:
#   • pin torch 2.4.0+cu121 to match the mmcv 2.2.0 wheel ABI,
#   • install the PREBUILT mmcv 2.2.0 (no compile, no nvcc),
#   • install mmpose (RTMPose) + mmdet, both --no-deps, patch version caps,
#   • pin a coherent torch-2.4-era diffusion stack:
#       diffusers 0.30.2 + transformers 4.44.2 + accelerate 0.34.2
# ════════════════════════════════════════════════════════════
import os, sys, re, glob, shutil, subprocess, time, tempfile
import urllib.request
import yaml as _yaml
from pathlib import Path

MT_REPO = Path(MODELS_DIR) / 'musetalk'
MT_OUT  = Path(OUTPUTS_DIR) / 'musetalk'
MT_OUT.mkdir(parents=True, exist_ok=True)

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

def _run(cmd, label, env=None, timeout=None, tail=4000):
    """Run a command, surfacing the REAL combined output tail on failure."""
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=timeout)
    if r.returncode != 0:
        print(f'  ✗ {label} FAILED ({_fmt(time.time()-t0)})', flush=True)
        combined = ((r.stdout or '') + '\n' + (r.stderr or '')).strip()
        print(f'  ── output tail ──\n{combined[-tail:]}\n  ────────────────', flush=True)
        raise RuntimeError(f'{label} failed — see output tail above.')
    print(f'  ✓ {label} ({_fmt(time.time()-t0)})', flush=True)
    return r

_PIP = [sys.executable, '-m', 'pip', 'install']

# ── Python 3.12 pkg_resources fix (root cause) ────────────────
# Overwrite Colab's ancient /usr/local pkg_resources (full of 3.12-removed
# APIs) by force-reinstalling a setuptools that still bundles a clean one.
def _ensure_pkg_resources(tag=''):
    v = subprocess.run([sys.executable, '-c', 'import pkg_resources'],
                       capture_output=True, text=True)
    if v.returncode == 0:
        return True
    _run(_PIP + ['-q', '--force-reinstall', '--no-cache-dir', 'setuptools==75.8.0'],
         f'reinstall setuptools 75.8.0 (fix pkg_resources){tag}')
    v = subprocess.run([sys.executable, '-c', 'import pkg_resources; print("pkg_resources OK")'],
                       capture_output=True, text=True)
    if v.returncode != 0:
        print(f'  pkg_resources STILL broken:\n{v.stderr[-600:]}')
        return False
    print(f'  {v.stdout.strip()}')
    return True

if not _ensure_pkg_resources():
    raise RuntimeError('Could not repair pkg_resources — cannot proceed.')
print(f'[{time.strftime("%H:%M:%S")}] pkg_resources OK.')

def _patch_mmcv_caps():
    """mmpose/mmdet cap mmcv < 2.2.0; we run mmcv 2.2.0. Bump the cap so
    their import-time asserts pass."""
    for pkg in ['mmpose', 'mmdet']:
        for init in glob.glob(f'/usr/local/lib/python3.*/dist-packages/{pkg}/__init__.py'):
            t = Path(init).read_text()
            t2 = re.sub(r"mmcv_maximum_version\s*=\s*'[^']+'",
                        "mmcv_maximum_version = '2.3.0'", t)
            if t2 != t:
                Path(init).write_text(t2)
                print(f'  Patched {pkg} mmcv_maximum_version → 2.3.0')

# Coherent torch-2.4-era diffusion stack (see header).
_DIFFUSION_PINS = ['diffusers==0.30.2', 'transformers==4.44.2', 'accelerate==0.34.2']
_DIFFUSION_TAG  = '0.30.2 4.44.2 0.34.2'

# ── Setup ──────────────────────────────────────────────────────
if not (MT_REPO / '.setup_done').exists():
    _step('SETUP — cloning MuseTalk + installing deps')
    if not MT_REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/TMElyralab/MuseTalk', str(MT_REPO)], check=True)

    # Core Python deps. Diffusion stack pinned to a coherent torch-2.4 set
    # (see header). Skip the numpy/tensorflow pins (conflict with torch).
    _run(_PIP + ['-q'] + _DIFFUSION_PINS + [
                 'omegaconf', 'einops', 'librosa',
                 'imageio', 'imageio-ffmpeg', 'av',
                 'soundfile', 'ffmpeg-python', 'gdown'], 'core deps')

    # ── OpenMMLab via PREBUILT wheels (no compile, no nvcc) ───
    _step('SETUP — OpenMMLab (torch 2.4.0 + prebuilt mmcv 2.2.0 + mmpose/mmdet)')

    # 1. Pin torch 2.4.0+cu121 to match the prebuilt mmcv 2.2.0 cp312 ABI.
    _run(_PIP + ['-q', 'torch==2.4.0', 'torchvision==0.19.0', 'torchaudio==2.4.0',
                 '--index-url', 'https://download.pytorch.org/whl/cu121'],
         'torch 2.4.0+cu121 (match mmcv wheel)')

    # 2. openmim + mmengine (pure Python)
    _run(_PIP + ['-q', '-U', 'openmim'], 'openmim')
    _ensure_pkg_resources(' [pre-mim]')
    _run([sys.executable, '-m', 'mim', 'install', 'mmengine'], 'mmengine')

    # 3. PREBUILT mmcv 2.2.0 cp312 wheel from the openmmlab index — no compile.
    _run(_PIP + ['-q', 'mmcv==2.2.0', '-f',
                 'https://download.openmmlab.com/mmcv/dist/cu121/torch2.4.0/index.html'],
         'mmcv 2.2.0 (prebuilt cp312 wheel)')

    # 4. mmpose + mmdet, both --no-deps (so pip won't downgrade mmcv), then
    #    install their runtime deps explicitly. mmdet is pure-Python (no
    #    compile) and only needed to satisfy mmpose's eager RTMO-head import.
    _run(_PIP + ['-q', '--no-deps', 'mmpose==1.3.2'], 'mmpose 1.3.2 (no-deps)')
    _run(_PIP + ['-q', '--no-deps', 'mmdet==3.3.0'], 'mmdet 3.3.0 (no-deps)')
    _run(_PIP + ['-q', 'json_tricks', 'munkres', 'xtcocotools', 'scipy',
                 'opencv-python', 'pycocotools', 'terminaltables', 'shapely',
                 'six', 'addict', 'yapf'], 'mmpose/mmdet runtime deps')
    _patch_mmcv_caps()

    # 5. Verify the whole OpenMMLab chain imports (ops + mmpose API)
    _v = subprocess.run([sys.executable, '-c',
                         'import mmcv; from mmcv.ops import RoIAlign; '
                         'from mmpose.apis import init_model, inference_topdown; '
                         'import torch; print("OpenMMLab OK | mmcv", mmcv.__version__, '
                         '"| torch", torch.__version__)'],
                        capture_output=True, text=True)
    if _v.returncode != 0:
        print(f'  ✗ OpenMMLab import check FAILED:\n{_v.stderr[-1500:]}')
        raise RuntimeError('OpenMMLab stack imported with errors — see tail above.')
    print(f'  ✓ {_v.stdout.strip()}')

    # MuseTalk scripts use moviepy.editor (→ pygame → pkg_resources crash).
    for _py in MT_REPO.rglob('*.py'):
        try: _t = _py.read_text()
        except Exception: continue
        if 'from moviepy.editor import' in _t:
            _t = _t.replace('from moviepy.editor import VideoFileClip, AudioFileClip',
                            'from moviepy.video.io.VideoFileClip import VideoFileClip\n'
                            'from moviepy.audio.io.AudioFileClip import AudioFileClip')
            _t = _t.replace('from moviepy.editor import VideoFileClip',
                            'from moviepy.video.io.VideoFileClip import VideoFileClip')
            _t = _t.replace('from moviepy.editor import AudioFileClip',
                            'from moviepy.audio.io.AudioFileClip import AudioFileClip')
            _py.write_text(_t)
            print(f'  Patched moviepy.editor import in {_py.relative_to(MT_REPO)}')

    (MT_REPO / '.setup_done').touch()
    print('  Setup done.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Diffusion stack compat with pinned torch 2.4.0 ────────────
# Runs every session (even when setup is cached): the latest transformers
# uses torch.float8_e8m0fnu (torch 2.5+ only); accelerate must be >=0.34 for
# diffusers' clear_device_cache. Pin the whole coherent set.
_dv = subprocess.run([sys.executable, '-c',
                      'import diffusers, transformers, accelerate; '
                      'print(diffusers.__version__, transformers.__version__, accelerate.__version__)'],
                     capture_output=True, text=True).stdout.strip()
if _dv != _DIFFUSION_TAG:
    _run(_PIP + ['-q'] + _DIFFUSION_PINS,
         f'pin diffusion stack {_DIFFUSION_TAG} (was: {_dv or "n/a"})')
else:
    print(f'[{time.strftime("%H:%M:%S")}] diffusion stack already pinned ({_DIFFUSION_TAG}).')

# ── Download weights (~10 GB total) ────────────────────────────
if not (MT_REPO / '.weights_done').exists():
    from huggingface_hub import snapshot_download
    _step('WEIGHTS — downloading MuseTalk models from HuggingFace')
    models_dir = MT_REPO / 'models'
    models_dir.mkdir(exist_ok=True)
    # NOTE: TMElyralab/MuseTalk downloads to models/ ROOT — the repo already
    # contains musetalk/ and musetalkV15/ subfolders, so local_dir must be
    # models/ (not models/musetalkV15) or the files get double-nested.
    for hf_id, local_name in [
        ('TMElyralab/MuseTalk',          '.'),
        ('stabilityai/sd-vae-ft-mse',    'sd-vae'),
        ('openai/whisper-tiny',          'whisper'),
        ('yzd-v/DWPose',                 'dwpose'),
    ]:
        dest = models_dir if local_name == '.' else models_dir / local_name
        marker = dest / ('musetalkV15' if local_name == '.' else '')
        print(f'  {hf_id} → {dest} ...', end=' ', flush=True)
        t0 = time.time()
        snapshot_download(hf_id, local_dir=str(dest),
                          ignore_patterns=['*.ckpt', 'flax*', 'tf_*'])
        print(f'done in {_fmt(time.time()-t0)}')
    (MT_REPO / '.weights_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Weights already downloaded, skipping.')

# ── Face-parsing weights (separate sources, not in the HF bundle) ──
# blending.py loads ./models/face-parse-bisent/{resnet18-5c106cde,79999_iter}.
# Idempotent: runs every session, downloads only what's missing.
_fpb     = MT_REPO / 'models' / 'face-parse-bisent'
_resnet  = _fpb / 'resnet18-5c106cde.pth'
_bisenet = _fpb / '79999_iter.pth'
_fpb.mkdir(parents=True, exist_ok=True)
if not _resnet.exists():
    _step('FACE-PARSE — downloading resnet18 backbone (PyTorch zoo)')
    urllib.request.urlretrieve('https://download.pytorch.org/models/resnet18-5c106cde.pth',
                               str(_resnet))
    print(f'  resnet18-5c106cde.pth ({_resnet.stat().st_size//1024//1024} MB)')
if not _bisenet.exists():
    _step('FACE-PARSE — downloading BiSeNet 79999_iter.pth (Google Drive via gdown)')
    import gdown
    gdown.download(id='154JgKpzCPW82qINcVieuPH3fZ2e0P812',
                   output=str(_bisenet), quiet=False)
    print(f'  79999_iter.pth ({_bisenet.stat().st_size//1024//1024} MB)')
print(f'[{time.strftime("%H:%M:%S")}] face-parse-bisent ready.')

# ── TTS driver audio ───────────────────────────────────────────
_DRIVER_WAV = '/tmp/sadtalker_driver.wav'
if not os.path.exists(_DRIVER_WAV):
    _step('TTS DRIVER — generating speech with edge-tts')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts', 'nest_asyncio'], check=True)
    import asyncio, nest_asyncio, edge_tts
    nest_asyncio.apply()
    async def _gen():
        await edge_tts.Communicate(
            "Hello, I am your virtual assistant. I am happy to help you today. "
            "Please feel free to ask me any questions you may have. "
            "I will do my best to assist you.", 'en-US-JennyNeural').save('/tmp/mt_drv.mp3')
    asyncio.get_event_loop().run_until_complete(_gen())
    subprocess.run(['ffmpeg', '-y', '-i', '/tmp/mt_drv.mp3',
                    '-ar', '16000', '-ac', '1', _DRIVER_WAV],
                   capture_output=True, check=True)
    print(f'  Driver audio ready.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Reusing driver audio.')

# ── Inference ──────────────────────────────────────────────────
_step(f'INFERENCE — {len(headshots)} headshots  |  GPU  |  MuseTalk v1.5')
total_start = time.time()
audio_dur   = os.path.getsize(_DRIVER_WAV) / (16000 * 2)

# Locate unet.pth robustly (the HF repo may nest musetalkV15/), take its
# JSON config from the same directory.
_unet_hits = glob.glob(str(MT_REPO / 'models' / '**' / 'unet.pth'), recursive=True)
if not _unet_hits:
    raise RuntimeError(f'unet.pth not found under {MT_REPO/"models"} — '
                       'delete .weights_done and re-run to re-download.')
_unet_path = _unet_hits[0]
_unet_dir  = Path(_unet_path).parent
_cfg_hits  = sorted(_unet_dir.glob('*.json'))
_unet_config = str(_cfg_hits[0]) if _cfg_hits else str(_unet_dir / 'musetalk.json')
print(f'  unet:   {_unet_path}')
print(f'  config: {_unet_config}')

for headshot in headshots:
    name     = Path(headshot).stem
    out_path = str(MT_OUT / f'{name}_talking.mp4')
    if os.path.exists(out_path):
        print(f'  skip {name} (already exists)')
        continue

    # MuseTalk needs a video input (same as Wav2Lip)
    tmp_vid = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
    subprocess.run(['ffmpeg', '-y', '-loop', '1', '-i', headshot,
                    '-t', str(audio_dur),
                    '-vf', 'fps=25,scale=trunc(iw/2)*2:trunc(ih/2)*2',
                    '-pix_fmt', 'yuv420p', tmp_vid],
                   capture_output=True, check=True)

    # Per-headshot inference config YAML
    cfg_path = f'/tmp/{name}_musetalk.yaml'
    _yaml.dump({'task_0': {'video_path': tmp_vid, 'audio_path': _DRIVER_WAV}},
               open(cfg_path, 'w'))

    result_dir = str(MT_REPO / 'results' / name)

    cmd = [sys.executable, '-m', 'scripts.inference',
           '--inference_config', cfg_path,
           '--result_dir',       result_dir,
           '--unet_model_path',  _unet_path,
           '--unet_config',      _unet_config,
           '--version',          'v15',
           '--ffmpeg_path',      '/usr/bin/ffmpeg']

    print(f'  [{time.strftime("%H:%M:%S")}] {name} ...', end=' ', flush=True)
    t0     = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(MT_REPO))
    elapsed = time.time() - t0

    try: os.unlink(tmp_vid)
    except OSError: pass

    if result.returncode != 0:
        print(f'ERROR\n{result.stderr[-1000:]}')
        continue

    candidates = glob.glob(os.path.join(result_dir, '**', '*.mp4'), recursive=True)
    if candidates:
        shutil.copy2(max(candidates, key=os.path.getmtime), out_path)
        print(f'done in {_fmt(elapsed)}  ({os.path.getsize(out_path)/1024/1024:.2f} MB)')
    else:
        print(f'WARNING: no output found in {result_dir}')

print(f'\n{"="*50}')
print(f'MuseTalk done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {MT_OUT}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — VideoReTalking  (3-stage: expression → lip sync → enhance)
# https://github.com/OpenTalker/video-retalking
# Same org as SadTalker — familiar tech stack, adds face enhancement
# ════════════════════════════════════════════════════════════
import os, sys, glob, shutil, subprocess, time, tempfile
import site
from pathlib import Path

VRT_REPO = Path(MODELS_DIR) / 'video-retalking'
VRT_OUT  = Path(OUTPUTS_DIR) / 'videoretalking'
VRT_OUT.mkdir(parents=True, exist_ok=True)

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

# ── Setup ──────────────────────────────────────────────────────
if not (VRT_REPO / '.setup_done').exists():
    _step('SETUP — cloning VideoReTalking + installing deps')
    if not VRT_REPO.exists():
        subprocess.run(['git', 'clone',
                        'https://github.com/OpenTalker/video-retalking', str(VRT_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'basicsr', 'facexlib', 'gfpgan',
                    'face-alignment', 'kornia==0.5.1',
                    'librosa', 'resampy', 'einops',
                    'scikit-image', 'imageio', 'imageio-ffmpeg'], check=True)
    # Patch basicsr for newer torchvision
    for sp in site.getsitepackages():
        target = Path(sp) / 'basicsr' / 'data' / 'degradations.py'
        if target.exists():
            txt = target.read_text(encoding='utf-8')
            OLD = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
            if OLD in txt:
                target.write_text(
                    txt.replace(OLD, 'from torchvision.transforms.functional import rgb_to_grayscale'),
                    encoding='utf-8')
                print('  Patched basicsr degradations.py')
            break
    (VRT_REPO / '.setup_done').touch()
    print('  Setup done.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Download checkpoints from HuggingFace ─────────────────────
if not (VRT_REPO / '.weights_done').exists():
    from huggingface_hub import snapshot_download
    _step('WEIGHTS — downloading VideoReTalking checkpoints from HuggingFace')
    print('  vinthony/video-retalking ...', end=' ', flush=True)
    t0 = time.time()
    snapshot_download('vinthony/video-retalking',
                      local_dir=str(VRT_REPO / 'checkpoints'))
    print(f'done in {_fmt(time.time()-t0)}')
    (VRT_REPO / '.weights_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Weights already downloaded, skipping.')

# ── TTS driver audio ───────────────────────────────────────────
_DRIVER_WAV = '/tmp/sadtalker_driver.wav'
if not os.path.exists(_DRIVER_WAV):
    _step('TTS DRIVER — generating speech with edge-tts')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts', 'nest_asyncio'], check=True)
    import asyncio, nest_asyncio, edge_tts
    nest_asyncio.apply()
    async def _gen():
        await edge_tts.Communicate(
            "Hello, I am your virtual assistant. I am happy to help you today. "
            "Please feel free to ask me any questions you may have. "
            "I will do my best to assist you.", 'en-US-JennyNeural').save('/tmp/vrt_drv.mp3')
    asyncio.get_event_loop().run_until_complete(_gen())
    subprocess.run(['ffmpeg', '-y', '-i', '/tmp/vrt_drv.mp3',
                    '-ar', '16000', '-ac', '1', _DRIVER_WAV],
                   capture_output=True, check=True)
    print(f'  Driver audio ready.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Reusing driver audio.')

# ── Inference ──────────────────────────────────────────────────
_step(f'INFERENCE — {len(headshots)} headshots  |  GPU  |  VideoReTalking')
total_start = time.time()
audio_dur   = os.path.getsize(_DRIVER_WAV) / (16000 * 2)

for headshot in headshots:
    name     = Path(headshot).stem
    out_path = str(VRT_OUT / f'{name}_talking.mp4')
    if os.path.exists(out_path):
        print(f'  skip {name} (already exists)')
        continue

    # VideoReTalking takes a video input, not a static image
    tmp_vid = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
    subprocess.run(['ffmpeg', '-y', '-loop', '1', '-i', headshot,
                    '-t', str(audio_dur),
                    '-vf', 'fps=25,scale=trunc(iw/2)*2:trunc(ih/2)*2',
                    '-pix_fmt', 'yuv420p', tmp_vid],
                   capture_output=True, check=True)

    cmd = [sys.executable, str(VRT_REPO / 'inference.py'),
           '--face',    tmp_vid,
           '--audio',   _DRIVER_WAV,
           '--outfile', out_path]

    print(f'  [{time.strftime("%H:%M:%S")}] {name} ...', end=' ', flush=True)
    t0     = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(VRT_REPO))
    elapsed = time.time() - t0

    try: os.unlink(tmp_vid)
    except OSError: pass

    if result.returncode != 0:
        print(f'ERROR\n{result.stderr[-600:]}')
        continue

    if os.path.exists(out_path):
        print(f'done in {_fmt(elapsed)}  ({os.path.getsize(out_path)/1024/1024:.2f} MB)')
    else:
        print(f'WARNING: no output at {out_path}')

print(f'\n{"="*50}')
print(f'VideoReTalking done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {VRT_OUT}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — AniPortrait  (ICLR 2025, diffusion, full audio-driven animation)
# https://github.com/Zejun-Yang/AniPortrait
# Audio → 3D pose + expression → diffusion video — eyes, head, lips all move
# ════════════════════════════════════════════════════════════
import os, sys, glob, shutil, subprocess, time
import yaml as _yaml
from pathlib import Path

ANI_REPO    = Path(MODELS_DIR) / 'aniportrait'
ANI_OUT     = Path(OUTPUTS_DIR) / 'aniportrait'
ANI_WEIGHTS = ANI_REPO / 'pretrained_weights'
ANI_OUT.mkdir(parents=True, exist_ok=True)

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

# ── Setup ──────────────────────────────────────────────────────
if not (ANI_REPO / '.setup_done').exists():
    _step('SETUP — cloning AniPortrait + installing deps')
    if not ANI_REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Zejun-Yang/AniPortrait', str(ANI_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'diffusers==0.24.0', 'transformers==4.30.2', 'accelerate==0.21.0',
                    'omegaconf', 'einops', 'decord', 'librosa',
                    'imageio', 'imageio-ffmpeg',
                    'controlnet-aux==0.0.7', 'mediapipe',
                    'insightface', 'onnxruntime-gpu'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', str(ANI_REPO / 'requirements.txt')], check=False)
    (ANI_REPO / '.setup_done').touch()
    print('  Setup done.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Download weights (~8 GB: SD-v1.5 + AniPortrait + wav2vec2) ─
if not (ANI_REPO / '.weights_done').exists():
    from huggingface_hub import snapshot_download
    _step('WEIGHTS — downloading AniPortrait + SD models (~8 GB total)')
    ANI_WEIGHTS.mkdir(exist_ok=True)
    for hf_id, local_name in [
        ('ZJYang/AniPortrait',                               'AniPortrait'),
        ('runwayml/stable-diffusion-v1-5',                   'stable-diffusion-v1-5'),
        ('stabilityai/sd-vae-ft-mse',                        'sd-vae-ft-mse'),
        ('lambdalabs/sd-image-variations-diffusers',         'sd-image-variations-diffusers'),
        ('facebook/wav2vec2-base-960h',                      'wav2vec2-base-960h'),
    ]:
        dest = ANI_WEIGHTS / local_name
        if not dest.exists():
            print(f'  {local_name} ...', end=' ', flush=True)
            t0 = time.time()
            snapshot_download(hf_id, local_dir=str(dest),
                              ignore_patterns=['*.ckpt', 'safety_checker*', 'flax*', 'tf_*'])
            print(f'done in {_fmt(time.time()-t0)}')
        else:
            print(f'  already have {local_name}')

    # AniPortrait stores weights in a subfolder — flatten to pretrained_weights root
    src = ANI_WEIGHTS / 'AniPortrait'
    if src.exists():
        for f in src.iterdir():
            dst = ANI_WEIGHTS / f.name
            if not dst.exists():
                (shutil.copytree if f.is_dir() else shutil.copy2)(str(f), str(dst))

    # image_encoder lives inside sd-image-variations-diffusers — symlink expected location
    img_enc_src = ANI_WEIGHTS / 'sd-image-variations-diffusers' / 'image_encoder'
    img_enc_dst = ANI_WEIGHTS / 'image_encoder'
    if img_enc_src.exists() and not img_enc_dst.exists():
        shutil.copytree(str(img_enc_src), str(img_enc_dst))

    (ANI_REPO / '.weights_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Weights already downloaded, skipping.')

# ── TTS driver audio ───────────────────────────────────────────
_DRIVER_WAV = '/tmp/sadtalker_driver.wav'
if not os.path.exists(_DRIVER_WAV):
    _step('TTS DRIVER — generating speech with edge-tts')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts', 'nest_asyncio'], check=True)
    import asyncio, nest_asyncio, edge_tts
    nest_asyncio.apply()
    async def _gen():
        await edge_tts.Communicate(
            "Hello, I am your virtual assistant. I am happy to help you today. "
            "Please feel free to ask me any questions you may have. "
            "I will do my best to assist you.", 'en-US-JennyNeural').save('/tmp/ani_drv.mp3')
    asyncio.get_event_loop().run_until_complete(_gen())
    subprocess.run(['ffmpeg', '-y', '-i', '/tmp/ani_drv.mp3',
                    '-ar', '16000', '-ac', '1', _DRIVER_WAV],
                   capture_output=True, check=True)
    print(f'  Driver audio ready.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Reusing driver audio.')

# ── Inference (one headshot at a time for clean output tracking) ─
_step(f'INFERENCE — {len(headshots)} headshots  |  GPU  |  AniPortrait audio2vid  |  -acc')
total_start = time.time()

for headshot in headshots:
    name     = Path(headshot).stem
    out_path = str(ANI_OUT / f'{name}_talking.mp4')
    if os.path.exists(out_path):
        print(f'  skip {name} (already exists)')
        continue

    # AniPortrait audio2vid config — test_cases is dict {image: [audio_list]}
    cfg_path = f'/tmp/{name}_aniportrait.yaml'
    _yaml.dump({
        'pretrained_base_model_path': str(ANI_WEIGHTS / 'stable-diffusion-v1-5'),
        'pretrained_vae_path':        str(ANI_WEIGHTS / 'sd-vae-ft-mse'),
        'image_encoder_path':         str(ANI_WEIGHTS / 'image_encoder'),
        'audio_inference_config':     str(ANI_REPO / 'configs' / 'inference' / 'inference_audio.yaml'),
        'inference_config':           str(ANI_REPO / 'configs' / 'inference' / 'inference_v2.yaml'),
        'save_dir':                   str(ANI_OUT),
        'test_cases':                 {headshot: [_DRIVER_WAV]},
    }, open(cfg_path, 'w'))

    cmd = [sys.executable, '-m', 'scripts.audio2vid',
           '--config', cfg_path,
           '-W', '512', '-H', '512',
           '-L', '300', '-acc']

    print(f'  [{time.strftime("%H:%M:%S")}] {name} ...', end=' ', flush=True)
    t0     = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ANI_REPO))
    elapsed = time.time() - t0

    if result.returncode != 0:
        print(f'ERROR\n{result.stderr[-600:]}')
        continue

    # AniPortrait writes {timestamp}_{cfg_name}_output.mp4 — find the newest
    candidates = sorted(glob.glob(str(ANI_OUT / '*.mp4')), key=os.path.getmtime, reverse=True)
    if candidates and candidates[0] != out_path:
        shutil.copy2(candidates[0], out_path)
        print(f'done in {_fmt(elapsed)}  ({os.path.getsize(out_path)/1024/1024:.2f} MB)')
    elif os.path.exists(out_path):
        print(f'done in {_fmt(elapsed)}  ({os.path.getsize(out_path)/1024/1024:.2f} MB)')
    else:
        print(f'WARNING: no output found in {ANI_OUT}')

print(f'\n{"="*50}')
print(f'AniPortrait done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {ANI_OUT}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — SONIC  (CVPR 2025, SVD-based, global audio perception)
# https://github.com/jixiaozhong/Sonic
# Stable Video Diffusion backbone — natural head + body motion from audio
# ════════════════════════════════════════════════════════════
import os, sys, glob, shutil, subprocess, time
from pathlib import Path

SONIC_REPO = Path(MODELS_DIR) / 'sonic'
SONIC_OUT  = Path(OUTPUTS_DIR) / 'sonic'
SONIC_OUT.mkdir(parents=True, exist_ok=True)

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

# ── Setup ──────────────────────────────────────────────────────
if not (SONIC_REPO / '.setup_done').exists():
    _step('SETUP — cloning SONIC + installing deps')
    if not SONIC_REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/jixiaozhong/Sonic', str(SONIC_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'diffusers==0.29.0', 'transformers==4.43.2', 'accelerate',
                    'omegaconf', 'librosa', 'einops',
                    'imageio', 'imageio-ffmpeg', 'tqdm'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', str(SONIC_REPO / 'requirements.txt')], check=False)
    (SONIC_REPO / '.setup_done').touch()
    print('  Setup done.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Download weights (~10 GB: SVD + SONIC + Whisper) ──────────
if not (SONIC_REPO / '.weights_done').exists():
    from huggingface_hub import snapshot_download
    _step('WEIGHTS — downloading SONIC + SVD checkpoints (~10 GB total)')
    ckpt_dir = SONIC_REPO / 'checkpoints'
    ckpt_dir.mkdir(exist_ok=True)
    for hf_id, local_name in [
        ('LeonJoe13/Sonic',                                  'Sonic'),
        ('stabilityai/stable-video-diffusion-img2vid-xt',   'stable-video-diffusion-img2vid-xt'),
        ('openai/whisper-tiny',                              'whisper-tiny'),
    ]:
        dest = ckpt_dir / local_name
        if not dest.exists():
            print(f'  {local_name} ...', end=' ', flush=True)
            t0 = time.time()
            snapshot_download(hf_id, local_dir=str(dest),
                              ignore_patterns=['*.ckpt', 'flax*', 'tf_*'])
            print(f'done in {_fmt(time.time()-t0)}')
        else:
            print(f'  already have {local_name}')
    (SONIC_REPO / '.weights_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Weights already downloaded, skipping.')

# ── TTS driver audio ───────────────────────────────────────────
_DRIVER_WAV = '/tmp/sadtalker_driver.wav'
if not os.path.exists(_DRIVER_WAV):
    _step('TTS DRIVER — generating speech with edge-tts')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts', 'nest_asyncio'], check=True)
    import asyncio, nest_asyncio, edge_tts
    nest_asyncio.apply()
    async def _gen():
        await edge_tts.Communicate(
            "Hello, I am your virtual assistant. I am happy to help you today. "
            "Please feel free to ask me any questions you may have. "
            "I will do my best to assist you.", 'en-US-JennyNeural').save('/tmp/sonic_drv.mp3')
    asyncio.get_event_loop().run_until_complete(_gen())
    subprocess.run(['ffmpeg', '-y', '-i', '/tmp/sonic_drv.mp3',
                    '-ar', '16000', '-ac', '1', _DRIVER_WAV],
                   capture_output=True, check=True)
    print(f'  Driver audio ready.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Reusing driver audio.')

# ── Inference ──────────────────────────────────────────────────
_step(f'INFERENCE — {len(headshots)} headshots  |  GPU  |  SONIC')
total_start = time.time()

for headshot in headshots:
    name     = Path(headshot).stem
    out_path = str(SONIC_OUT / f'{name}_talking.mp4')
    if os.path.exists(out_path):
        print(f'  skip {name} (already exists)')
        continue

    cmd = [sys.executable, str(SONIC_REPO / 'demo.py'),
           headshot, _DRIVER_WAV, out_path,
           '--seed', '42']

    print(f'  [{time.strftime("%H:%M:%S")}] {name} ...', end=' ', flush=True)
    t0     = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(SONIC_REPO))
    elapsed = time.time() - t0

    if result.returncode != 0:
        print(f'ERROR\n{result.stderr[-600:]}')
        continue

    if os.path.exists(out_path):
        print(f'done in {_fmt(elapsed)}  ({os.path.getsize(out_path)/1024/1024:.2f} MB)')
    else:
        # SONIC might write to a subdirectory — find and move it
        candidates = glob.glob(str(SONIC_REPO / 'outputs' / f'**/{name}*.mp4'), recursive=True)
        if candidates:
            shutil.copy2(max(candidates, key=os.path.getmtime), out_path)
            print(f'done in {_fmt(elapsed)}  (found in outputs/)')
        else:
            print(f'WARNING: no output found — check {SONIC_REPO}/outputs/')

print(f'\n{"="*50}')
print(f'SONIC done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {SONIC_OUT}')